# 4 · Lojistik Regresyon
Default benzeri veri: fit, olasılık, eşik, confusion matrix, ROC/AUC, multi-class, polinom özellikler.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(0)

In [2]:
# Default benzeri sentetik veri (gerçek Default.csv varsa onu oku)
n = 10000
student = np.random.binomial(1, 0.29, n)
balance = np.clip(np.random.normal(800 + 300 * student, 480, n), 0, None)
income = np.clip(np.random.normal(40 - 22 * student, 10, n), 1, None)
logit_p = -10.869 + 0.00574 * balance + 0.003 * income - 0.6468 * student
default = np.random.binomial(1, 1 / (1 + np.exp(-logit_p)))
df = pd.DataFrame({"balance": balance, "income": income, "student": student, "default": default})
print(df["default"].mean())   # ~%3 temerrüt -> dengesiz veri

0.037


In [3]:
import statsmodels.formula.api as smf
print(smf.logit("default ~ balance", data=df).fit(disp=0).summary().tables[1])
print(smf.logit("default ~ student", data=df).fit(disp=0).params)              # student + çıkar
m = smf.logit("default ~ balance + income + student", data=df).fit(disp=0)
print(m.summary().tables[1])                                                   # student - çıkar (confounding)
print("Odds ratio'lar:\n", np.exp(m.params).round(4))

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -10.9629      0.363    -30.168      0.000     -11.675     -10.251
balance        0.0056      0.000     25.765      0.000       0.005       0.006
Intercept   -3.518583
student      0.718350
dtype: float64
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -11.3722      0.467    -24.364      0.000     -12.287     -10.457
balance        0.0059      0.000     25.618      0.000       0.005       0.006
income         0.0059      0.007      0.870      0.384      -0.007       0.019
student       -0.6509      0.203     -3.207      0.001      -1.049      -0.253
Odds ratio'lar:
 Intercept    0.0000
balance      1.0060
income       1.0059
student      0.5216
dtype: float64


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X, y = df[["balance", "income", "student"]], df["default"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1, stratify=y)
logit = LogisticRegression(C=1e10, max_iter=5000).fit(X_train, y_train)   # C çok büyük = regularization yok (MLE)
print(logit.intercept_, logit.coef_)

# Elle tahmin: student, balance 1500, income 40
z = logit.intercept_[0] + logit.coef_[0] @ np.array([1500, 40, 1])
print("p =", 1 / (1 + np.exp(-z)), " sklearn:", logit.predict_proba(pd.DataFrame([[1500, 40, 1]], columns=X.columns))[0, 1])

[-11.41367111] [[ 0.00591022  0.00780246 -0.58839216]]
p = 0.056007470488305976  sklearn: 0.056007470488305976


In [5]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
p = logit.predict_proba(X_test)[:, 1]
for t in [0.5, 0.2]:
    yhat = (p > t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yhat).ravel()     # satır = gerçek
    print(f"eşik {t}: TN={tn} FP={fp} FN={fn} TP={tp} | hata={(fp+fn)/len(y_test):.4f} "
          f"sens={tp/(tp+fn):.3f} spec={tn/(tn+fp):.3f}")
print(classification_report(y_test, (p > 0.5).astype(int)))

eşik 0.5: TN=1918 FP=8 FN=54 TP=20 | hata=0.0310 sens=0.270 spec=0.996
eşik 0.2: TN=1867 FP=59 FN=31 TP=43 | hata=0.0450 sens=0.581 spec=0.969
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      1926
           1       0.71      0.27      0.39        74

    accuracy                           0.97      2000
   macro avg       0.84      0.63      0.69      2000
weighted avg       0.96      0.97      0.96      2000



In [6]:
fpr, tpr, thr = roc_curve(y_test, p)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, p):.3f}")
plt.plot([0, 1], [0, 1], "--", c="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(); plt.show()

## Multi-class ve lineer olmayan karar sınırı

In [7]:
from sklearn.datasets import load_iris, make_circles
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
Xi, yi = load_iris(return_X_y=True)
mc = LogisticRegression(max_iter=1000).fit(Xi, yi)       # multinomial (softmax)
print(mc.predict_proba(Xi[[0, 60, 120]]).round(3))

Xc, yc = make_circles(400, noise=0.15, factor=0.4, random_state=0)
lin = LogisticRegression().fit(Xc, yc)
poly = make_pipeline(PolynomialFeatures(2), StandardScaler(), LogisticRegression()).fit(Xc, yc)
print("Lineer sınır acc:", lin.score(Xc, yc), " | Polinom (derece 2) acc:", poly.score(Xc, yc))

[[0.982 0.018 0.   ]
 [0.056 0.938 0.007]
 [0.    0.024 0.976]]
Lineer sınır acc: 0.4975  | Polinom (derece 2) acc: 0.995
